# Karras Pre-conditioning (EDM)

## Complete Beginner's Guide

This notebook implements techniques from the influential paper **"Elucidating the Design Space of Diffusion-Based Generative Models"** (Karras et al., 2022), commonly called the **EDM paper**.

### What You'll Learn

1. **Karras Pre-conditioning**: A better way to train diffusion models
2. **Sigma-based Formulation**: Using noise level (sigma) instead of time
3. **Log-normal Sigma Sampling**: Better distribution for training
4. **Karras Sigma Schedule**: Optimal noise levels for sampling
5. **Multiple Samplers**: Euler, Heun, Ancestral, and LMS methods

### Prerequisites
- Understanding of DDPM and DDIM from previous notebooks
- Basic calculus concepts (optional but helpful)

### Why Pre-conditioning Matters

In standard diffusion models, the network's job changes dramatically based on the noise level:
- At low noise: Network should output something close to input
- At high noise: Network should output something very different from input

**Pre-conditioning** normalizes the inputs and targets so the network always operates in a similar regime, making training more stable and efficient.

---
## Setup and Imports

In [ ]:
# Select GPU
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
# Main imports
import timm, torch, random, datasets, math
import fastcore.all as fc
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import k_diffusion as K  # Katherine Crowson's k-diffusion library
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F

from torch.utils.data import DataLoader, default_collate
from pathlib import Path
from torch.nn import init
from fastcore.foundation import L
from torch import nn, tensor
from datasets import load_dataset
from operator import itemgetter
from torcheval.metrics import MulticlassAccuracy
from functools import partial
from torch.optim import lr_scheduler
from torch import optim

# miniai modules
from miniai.datasets import *
from miniai.conv import *
from miniai.learner import *
from miniai.activations import *
from miniai.init import *
from miniai.sgd import *
from miniai.resnet import *
from miniai.augment import *
from miniai.accel import *

In [ ]:
# Progress bar and diffusers
from fastprogress import progress_bar
from diffusers import UNet2DModel, DDIMPipeline, DDPMPipeline, DDIMScheduler, DDPMScheduler

In [ ]:
# Configure display
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
mpl.rcParams['figure.dpi'] = 70

import logging
logging.disable(logging.WARNING)

set_seed(42)
if fc.defaults.cpus > 8:
    fc.defaults.cpus = 8

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATIONS -- setup (run this cell ONCE).
# Each "🎮 Interactive" cell below loads a standalone HTML file from the
# published copy on GitHub Pages, in an isolated <iframe> (its CSS/JS can't leak
# into the notebook). It is shown FULL WIDTH and auto-fits its content height.
# No local files needed -- the visualizations are read from GitHub.
# ============================================================================
from IPython.display import HTML

def show_viz(path, height="600px"):
    """Embed an interactive visualization full width; it auto-fits its height.
    `path` may be a bare 'interactive_viz/<file>.html' (resolved to the GitHub
    Pages copy) or a full https URL."""
    base = "https://shammun.github.io/shammunul-fastai-notes/notebooks/"
    if not path.startswith("http"):
        path = base + path
    return HTML(
        f'<iframe src="{path}" loading="lazy" allowfullscreen '
        f'style="width:100%;height:{height};border:1px solid #dde5f2;border-radius:12px;'
        f'box-shadow:0 8px 24px rgba(123,92,214,.12);background:#fff;"></iframe>'
        '<script>addEventListener("message",function(e){'
        'if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){'
        'var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){'
        'if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>'
    )

---
## Load Dataset

In [ ]:
# Dataset configuration
xl, yl = 'image', 'label'
name = "zalando-datasets/fashion_mnist"
n_steps = 1000
bs = 512

# Load dataset
dsd = load_dataset(name)

In [ ]:
# Transform: pad to 32x32 and scale to [-1, 1]
@inplace
def transformi(b):
    b[xl] = [F.pad(TF.to_tensor(o), (2,2,2,2)) * 2 - 1 for o in b[xl]] # Now between -1 and 1

# Apply transform
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs)

# Get a sample batch
dl = dls.train
xb, yb = b = next(iter(dl))

---
## Sigma_data: The Data Standard Deviation

A key concept in Karras pre-conditioning is **sigma_data** (σ_data) - the standard deviation of your training data. This helps normalize the scales appropriately.

In [ ]:
# We could compute this from data:
# sig_data = xb.std()

# But often a fixed value works well
# For images scaled to [-1, 1], typical values are 0.5-1.0
sig_data = 0.66 # But Jeremy found better result with 0.33

**What is sigma_data?**

- σ_data represents the "typical magnitude" of your clean data
- For images in [-1, 1], it's around 0.5-1.0
- It's used to balance the signal and noise contributions

The EDM paper shows that proper scaling relative to σ_data significantly improves training stability.

# Jeremy's explanation of the Karras paper

Sometimes may be predicting the noise is a bad idea. So like you can either try and predict the noise or you can try and predict the clean image and each of those can be a b better idea in different stuations. If you are given something which are pure noise (the model has given something which is nearly pure noise) and is then asked to predict the noise, that's basically a waste of time because the whole thing is noise. 

If we do the opposite, which is try to get it predict the clean image, then if we give it a clean image that's nearly clean and try to predict the clean image, that's nearly a waste of time as well. So we want something which is like, regardless of how noisy the image is, we want it to be like an equally difficult problem to solve. 

So, what Karras do is they basically use a new thing called `c_skip` which is a number that says something like "for the training set, we should not just predict the noise all the time, not just predict clean image all the time, but predict kind of lerp version of one or other depending on how noisy it is.

In the equation shown below, y is the plain image, n is the noise. So `y + n` is the noised image. So, if `c_skip` was 0, then we would be predicting the clean image. If `c_skip` was 1, we would be predicting  y - (y + n)` or `n` or just noise. So, we can decide by picking a different `c_skip` whether you are predicting clean image or noise. 

They have made this `c_skip` a function of `sigma`. Unfortunately, `sigma` is the same thing as `alpha_bar` used to be. 

### 🎮 Interactive: Why Pre-conditioning? Feel the Prediction Problem

Jeremy's argument above is the heart of the whole paper, so before any math, **play with it**. Below, a tiny smiley is
noised at a σ you control, and the question is *what should the network be asked to predict?*

- Chips **①–⑤** walk the argument: the setup, then **target = ε** (watch the copy-paste meter — at high σ the noisy
  input practically *is* the noise, so echoing the input scores near-perfectly and nothing useful is learned), then
  **target = x₀** (same waste at the *other* extreme), then **EDM's blended target**, whose shortcut meter reads **0 at
  every σ** — no free lunch anywhere.
- Chip **⑤ sweeps σ** across the learning-signal plot; you can also drag the ✏️ dot yourself.
- Click any panel in the assembly row to interrogate it — the matching line of code lights up, and the rest of the code
  stays visible for context.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_why_precond.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/karras_why_precond.html", height="1000px")

---
## Karras Pre-conditioning Scalings

The EDM paper derives optimal scaling factors for:
1. **c_skip**: How much of the input to pass through unchanged
2. **c_out**: How to scale the network output
3. **c_in**: How to scale the network input

These ensure the network always operates in a well-conditioned regime.

![](img_3.png)

### Deriving the Effective Training Loss of EDM (Equation 8)

The image shows the inner term of the EDM training loss — the quantity that the neural network $F_\theta$ is trained to minimize:

$$\left\| \underbrace{F_\theta\!\bigl(c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n});\; c_{\text{noise}}(\sigma)\bigr)}_{\text{network output}} - \underbrace{\frac{1}{c_{\text{out}}(\sigma)}\bigl(\mathbf{y} - c_{\text{skip}}(\sigma)\cdot(\mathbf{y}+\mathbf{n})\bigr)}_{\text{effective training target}} \right\|_2^2$$

This is the core of **Equation 8** in the paper. It reveals what the raw neural network $F_\theta$ actually sees and learns during training, after all the preconditioning layers are accounted for. Below we derive it step by step.

> **New to the notation?** $F_\theta$ is a U-Net neural network with learnable parameters $\theta$ (see Section 0.4). The expression $F_\theta\!\bigl(c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n});\; c_{\text{noise}}(\sigma)\bigr)$ is a function call with two arguments separated by a semicolon — the scaled noisy image and the noise-level conditioning — explained in detail in Section 0.4b. The semicolon notation is explained in Section 0.10.

---

## 0. Notation Guide — Reading the Math

Before diving into the derivation, let us carefully explain every piece of mathematical notation that appears in this document. If you understand this section, the rest of the math will read like plain English.

### 0.1 — The $\mathbb{E}$ Symbol: "Expected Value" (i.e., Average)

The symbol $\mathbb{E}$ stands for **expectation**, which is just a fancy word for **"average over all possibilities."** When you see $\mathbb{E}[\text{something}]$, it means: "compute that *something* for every possible input, then take the average."

**Concrete analogy:** If you roll a die and compute the square of the result, the expectation is:

$$\mathbb{E}[\text{roll}^2] = \frac{1}{6}(1^2 + 2^2 + 3^2 + 4^2 + 5^2 + 6^2) = 15.17$$

You average $\text{roll}^2$ over all possible rolls. The $\mathbb{E}$ symbol is doing exactly this, but for continuous random variables (images, noise) instead of dice.

### 0.2 — The Subscript on $\mathbb{E}$: "Average Over What?"

The subscript tells you **which random variable you are averaging over** and **where it comes from**:

$$\mathbb{E}_{\mathbf{y} \sim p_{\text{data}}}[\;\cdots\;]$$

This reads: *"Pick a random image $\mathbf{y}$ from the data distribution $p_{\text{data}}$ (your training set), compute $\cdots$, and average the result over all possible images."*

$$\mathbb{E}_{\mathbf{n} \sim \mathcal{N}(\mathbf{0},\,\sigma^2\mathbf{I})}[\;\cdots\;]$$

This reads: *"Pick a random noise vector $\mathbf{n}$ from a Gaussian distribution with mean zero and variance $\sigma^2$ in every pixel, compute $\cdots$, and average."*

The $\sim$ symbol means **"drawn from"** or **"sampled from."** So $\mathbf{y} \sim p_{\text{data}}$ means "$\mathbf{y}$ is a random sample from the training data distribution."

### 0.3 — Why TWO $\mathbb{E}$'s in the Loss?

In the first loss equation, you see **two** $\mathbb{E}$'s nested together:

$$L = \mathbb{E}_{\mathbf{y}\sim p_{\text{data}}}\;\mathbb{E}_{\mathbf{n}\sim\mathcal{N}(\mathbf{0},\,\sigma^2\mathbf{I})} \left[\;\cdots\;\right]$$

This means: **average over two independent random choices:**

1. **Outer $\mathbb{E}$:** Pick a random clean image $\mathbf{y}$ from the dataset
2. **Inner $\mathbb{E}$:** Pick a random noise vector $\mathbf{n}$

Then compute the thing inside $[\cdots]$ and average over **all possible combinations** of images and noise vectors.

**Tiny concrete example to build intuition.** Suppose your "dataset" is just 2 images ($\mathbf{y}_1$ = cat, $\mathbf{y}_2$ = dog), and for each image you try 2 noise patterns ($\mathbf{n}_a$, $\mathbf{n}_b$). The two expectations become:

$$\mathbb{E}_{\mathbf{y}}\mathbb{E}_{\mathbf{n}}[\text{error}] = \frac{1}{2}\Bigl(\underbrace{\frac{1}{2}\bigl(\text{error}(\mathbf{y}_1,\mathbf{n}_a) + \text{error}(\mathbf{y}_1,\mathbf{n}_b)\bigr)}_{\text{inner } \mathbb{E}_\mathbf{n} \text{ for cat}} + \underbrace{\frac{1}{2}\bigl(\text{error}(\mathbf{y}_2,\mathbf{n}_a) + \text{error}(\mathbf{y}_2,\mathbf{n}_b)\bigr)}_{\text{inner } \mathbb{E}_\mathbf{n} \text{ for dog}}\Bigr)$$

The inner $\mathbb{E}_\mathbf{n}$ averages over noise *for a fixed image*. The outer $\mathbb{E}_\mathbf{y}$ then averages those results *across images*. In the end, you average all 4 combinations — which is exactly what the compact $\mathbb{E}_{\mathbf{y},\mathbf{n}}$ would give.

**In practice (during actual training),** you don't compute the true average over every possible image and noise — that would be impossible. Instead, each training step **approximates** these two expectations by:
- Picking one random image $\mathbf{y}$ from a minibatch (approximates $\mathbb{E}_{\mathbf{y}}$)
- Generating one random noise vector $\mathbf{n}$ (approximates $\mathbb{E}_{\mathbf{n}}$)

Over many training steps, the random sampling covers the full distributions, and the running average of the loss converges to the true $\mathbb{E}$.

**Why not write just one $\mathbb{E}$?** You could! The compact form $\mathbb{E}_{\sigma,\mathbf{y},\mathbf{n}}[\cdots]$ (used in the second equation) is shorthand for nesting all three expectations. Writing them separately makes explicit which variable comes from which distribution.

### 0.4 — $D_\theta$ and $F_\theta$: What Are These? Why Two Networks?

#### What is $F_\theta$?

$F_\theta$ is the **actual neural network** — the thing with conv layers, attention layers, residual blocks, and millions of learnable weights. In the EDM paper, $F_\theta$ is specifically a **U-Net** architecture (either DDPM++ or ADM style), which is the standard workhorse for diffusion models. It takes in an image-shaped tensor, processes it through downsampling and upsampling blocks with skip connections, and outputs another image-shaped tensor of the same size.

The letter $F$ is just a name — you could call it $\text{Net}$, $\text{UNet}$, or anything. The subscript $\theta$ represents **all the learnable parameters** (weights and biases) of this neural network. In a network with, say, 62 million parameters, $\theta$ is one giant vector containing all 62 million numbers. These are the numbers that gradient descent updates during training.

#### What is $D_\theta$?

$D_\theta$ is the **full denoiser** — it's NOT a separate neural network. It is $F_\theta$ **wrapped in a preconditioning shell** made of simple, fixed arithmetic operations. Think of it like this:

$$D_\theta(\mathbf{x};\sigma) = \underbrace{c_{\text{skip}}(\sigma) \cdot \mathbf{x}}_{\text{fixed arithmetic (no learning)}} + \underbrace{c_{\text{out}}(\sigma)}_{\text{fixed scalar}} \cdot \underbrace{F_\theta\!\bigl(c_{\text{in}}(\sigma)\cdot\mathbf{x};\; c_{\text{noise}}(\sigma)\bigr)}_{\text{the actual neural network}}$$

$D_\theta$ is what the outside world interacts with: you give it a noisy image, it gives back a denoised image. But internally, $D_\theta$ is doing simple rescaling before and after calling $F_\theta$.

**Analogy:** Think of $F_\theta$ as a chef, and $D_\theta$ as the restaurant. The restaurant ($D_\theta$) handles everything: it takes the customer's order (noisy image), preps the ingredients (scales the input by $c_{\text{in}}$), sends them to the chef ($F_\theta$), plates the dish (scales the output by $c_{\text{out}}$), adds garnish (adds the skip connection $c_{\text{skip}} \cdot \mathbf{x}$), and serves it to the customer. The chef ($F_\theta$) is the only one who actually *learns* — everything else is fixed recipe steps.

#### Why do we need BOTH?

The loss function is naturally written in terms of $D_\theta$ (because the goal is to denoise). But training happens to $F_\theta$ (because that's where the learnable weights are). The entire derivation in this document transforms the $D_\theta$ loss into an $F_\theta$ loss, revealing what $F_\theta$ actually needs to learn.

**Why write $\theta$ at all?** To emphasize that $D$ and $F$ are not fixed mathematical formulas — they are functions whose behavior **changes as training updates $\theta$**. Writing $D_\theta$ reminds us that this is *our model's approximation*, not the ideal denoiser (without subscript) that would perfectly denoise any image.

### 0.4b — Dissecting $F_\theta\!\bigl(c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n});\; c_{\text{noise}}(\sigma)\bigr)$

This expression looks dense, but it's just a function call with two arguments. Let's take it apart piece by piece:

$$F_\theta\!\Bigl(\;\underbrace{c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n})}_{\text{argument 1: the image input}}\;;\;\underbrace{c_{\text{noise}}(\sigma)}_{\text{argument 2: the conditioning input}}\;\Bigr)$$

**In Python, this would be:**

```python
# Build argument 1: the scaled noisy image
noisy_image = y + n                        # add noise to clean image
scaled_input = c_in(sigma) * noisy_image   # normalize to unit variance

# Build argument 2: the noise level encoding
noise_conditioning = c_noise(sigma)        # = (1/4) * ln(sigma)

# Call the neural network
output = F_theta(scaled_input, noise_conditioning)
```

**Walking through each piece:**

**Piece 1 — $(\mathbf{y}+\mathbf{n})$:** This is the noisy image. $\mathbf{y}$ is a clean image from the training set, and $\mathbf{n}$ is random Gaussian noise. Adding them gives a corrupted image that the denoiser must clean up. At $\sigma = 0.002$, the noise is barely visible; at $\sigma = 80$, the image is completely buried in noise.

**Piece 2 — $c_{\text{in}}(\sigma) \cdot (\mathbf{y}+\mathbf{n})$:** The noisy image gets **multiplied by a scalar** $c_{\text{in}}(\sigma)$ before entering the network. This is just a brightness scaling — every pixel gets multiplied by the same number. The purpose: normalize the input so its variance is always 1, regardless of $\sigma$. Without this, the network would see inputs ranging from magnitude ~0.5 (low noise) to magnitude ~80 (high noise), making training unstable.

**Piece 3 — $c_{\text{noise}}(\sigma)$:** This is a **single number** (not an image) that tells the network "how noisy is the input you just received?" EDM uses $c_{\text{noise}}(\sigma) = \frac{1}{4}\ln(\sigma)$, which compresses the huge range $\sigma \in [0.002, 80]$ into a small range $\approx [-1.6, 1.1]$. Inside the network, this number gets expanded into an embedding vector (similar to positional embeddings in Transformers) and injected into every layer via adaptive normalization or addition.

**Piece 4 — The semicolon ";" between the two arguments:** See Section 0.10 below. In short: the semicolon separates two *fundamentally different kinds* of input. The first argument is spatial data (an image with height, width, channels). The second argument is metadata (a single number encoding the noise level). They enter the network through completely different pathways — the image goes through convolutions, while the conditioning scalar goes through an embedding layer and gets injected via normalization.

**Piece 5 — $F_\theta(\cdots)$:** The output is another image-shaped tensor, same dimensions as the input. This is what the network "thinks" the answer should be. Depending on $\sigma$, this answer represents different things (scaled noise at low $\sigma$, scaled clean image at high $\sigma$ — see Section 4).

**The full data flow, visualized:**

```
                            c_noise(σ) = ¼ ln(σ)
                                  │
                                  ▼
                          ┌──────────────┐
  y + n ──→ ×c_in(σ) ──→ │              │
            (normalize)   │   F_θ (UNet) │ ──→ network output
                          │              │     (image-shaped)
                          └──────────────┘
```

### 0.5 — $L(D_\theta;\,\sigma)$: What Does This Notation Mean?

$L$ is the **loss function** — a single number that measures "how bad is our model right now?"

- $L(D_\theta;\,\sigma)$: The loss of denoiser $D_\theta$ **at a specific noise level** $\sigma$. The semicolon separates the function being evaluated ($D_\theta$) from the condition ($\sigma$). It reads: *"The loss of model $D_\theta$, evaluated at noise level $\sigma$."*
- $L(D_\theta)$: The **total** loss of denoiser $D_\theta$, averaged over **all** noise levels. No semicolon because there's no specific $\sigma$ — we've averaged over all of them.

**Training goal:** Find the $\theta$ that makes $L(D_\theta)$ as small as possible.

### 0.6 — $\|\cdot\|_2^2$: Squared L2 Norm

$$\bigl\| D_\theta(\mathbf{y}+\mathbf{n};\,\sigma) - \mathbf{y} \bigr\|_2^2$$

This is the **sum of squared pixel-wise differences** — the most basic measure of "how different are these two images?"

If images have $d$ pixels (e.g., $d = 32 \times 32 \times 3 = 3072$ for CIFAR-10):

$$\|\mathbf{a} - \mathbf{b}\|_2^2 = \sum_{i=1}^{d} (a_i - b_i)^2$$

So $\|D_\theta(\mathbf{y}+\mathbf{n};\,\sigma) - \mathbf{y}\|_2^2$ means: *"For every pixel, compute (predicted pixel − true pixel)$^2$, and sum them all up."* A perfect denoiser would give 0.

### 0.7 — $\sigma$, $\sigma_{\text{data}}$, $p_{\text{data}}$, $p_{\text{train}}$: The Key Symbols

| Symbol | What it is | Typical value | Intuition |
|---|---|---|---|
| $\sigma$ | Noise level (standard deviation of added noise) | 0.002 to 80 | "How noisy is the input?" Higher $\sigma$ = more noise |
| $\sigma_{\text{data}}$ | Standard deviation of the clean training data | 0.5 | "How spread out are pixel values in the dataset?" For images in $[-1, 1]$, this is around 0.5 |
| $p_{\text{data}}$ | The training data distribution | — | The "true" distribution of clean images. In practice, your training set |
| $p_{\text{train}}(\sigma)$ | Distribution over noise levels during training | Log-normal | "How often do we train at each noise level?" EDM uses $\ln(\sigma) \sim \mathcal{N}(-1.2,\, 1.2^2)$ |
| $\lambda(\sigma)$ | Per-noise-level loss weight | $\frac{\sigma^2+\sigma_{\text{data}}^2}{(\sigma\,\sigma_{\text{data}})^2}$ | "How much should we care about errors at this noise level?" |

### 0.8 — $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$: Gaussian (Normal) Distribution

$\mathcal{N}(\mathbf{0},\,\sigma^2\mathbf{I})$ is a **multivariate Gaussian distribution** with:

- Mean $\mathbf{0}$ (centered at zero in every dimension/pixel)
- Covariance $\sigma^2\mathbf{I}$ (each pixel gets independent noise with the same variance $\sigma^2$)

The $\mathbf{I}$ is the **identity matrix**, meaning the noise in each pixel is independent of noise in every other pixel. Writing $\mathbf{n} \sim \mathcal{N}(\mathbf{0},\,\sigma^2\mathbf{I})$ means: *"Generate a noise image where each pixel is independently drawn from a normal distribution with mean 0 and standard deviation $\sigma$."*

### 0.9 — $\text{Var}[\cdot]$: Variance

$\text{Var}_{\mathbf{y},\mathbf{n}}[\cdots]$ means the **variance** of the quantity inside the brackets, computed over all possible values of $\mathbf{y}$ and $\mathbf{n}$. Variance measures "how spread out are the values?"

**Intuition:** If a random variable always gives roughly the same value, its variance is near 0. If it gives wildly different values each time, variance is large. Formally, $\text{Var}[X] = \mathbb{E}[(X - \mathbb{E}[X])^2]$ — the average squared deviation from the mean.

**Why we care about variance here:** The whole point of preconditioning is to ensure that the network input and training target both have variance = 1 (neither too big nor too small). If the target had variance 10000, the network would need to output huge numbers; if it had variance 0.0001, the network output would be drowning in floating-point noise. Variance = 1 is the "Goldilocks zone."

Key properties used in the derivations:
- $\text{Var}[\mathbf{y}] = \sigma_{\text{data}}^2$ (the spread of clean images — by definition, $\sigma_{\text{data}}$ is the standard deviation of the data)
- $\text{Var}[\mathbf{n}] = \sigma^2$ (the spread of noise — we chose to add noise with standard deviation $\sigma$)
- $\text{Var}[\mathbf{y} + \mathbf{n}] = \sigma_{\text{data}}^2 + \sigma^2$ (because $\mathbf{y}$ and $\mathbf{n}$ are **independent**, variances simply add — this is a fundamental theorem of probability)
- $\text{Var}[a \cdot \mathbf{x}] = a^2 \cdot \text{Var}[\mathbf{x}]$ (scaling by constant $a$ scales variance by $a^2$ — if you double all values, the spread quadruples)

### 0.10 — The Semicolon in $F_\theta(\mathbf{x};\,\sigma)$ and $D_\theta(\mathbf{x};\,\sigma)$

The semicolon ";" is **not a colon** — it is a **semicolon**, and in mathematical notation it separates **two fundamentally different kinds of inputs:**

- **Before the semicolon:** $\mathbf{x}$ — the **data input** (the noisy image to be processed)
- **After the semicolon:** $\sigma$ — the **conditioning input** (metadata about the noise level)

**Why not just use a comma?** A comma ($f(a, b)$) typically means both arguments play equal, interchangeable roles. The semicolon ($f(a;\, b)$) signals an asymmetry: *"$a$ is the thing being processed; $b$ is context that modifies how $a$ gets processed."*

**How this works mechanically inside the network:**

The two arguments enter through **completely different pathways**:

```
ARGUMENT 1 (before semicolon): c_in(σ) · (y+n) — an IMAGE
  → Shape: [batch, channels, height, width] e.g. [64, 3, 32, 32]
  → Enters through: the main convolutional pathway of the U-Net
  → Processed by: conv layers, attention layers, residual blocks
  → This is what the network "sees" as spatial data

ARGUMENT 2 (after semicolon): c_noise(σ) — a SINGLE NUMBER
  → Shape: [batch] e.g. [64] (one number per image in the batch)
  → Enters through: a separate embedding pathway:
      1. Passed through sinusoidal positional encoding (like in Transformers)
      2. Then through a small MLP to produce an embedding vector
      3. This vector is injected into EVERY layer of the U-Net
         via adaptive group normalization: scale and shift the features
  → This tells each layer "the input has noise level σ, adjust accordingly"
```

**Analogy:** Think of $\mathbf{x}$ as a photo you hand to a photo editor, and $\sigma$ as a sticky note on the photo that says "this photo has heavy grain, level 80." The editor (network) looks at the photo through their normal visual processing (conv layers) but reads the sticky note through a completely different channel (embedding pathway) to decide *how* to edit it.

**In Python, the semicolon maps to two separate function arguments:**

```python
# The semicolon in F_θ(x; σ) becomes two Python arguments:
class UNet(nn.Module):
    def forward(self, x, sigma_embedding):
        #        ↑ argument 1     ↑ argument 2
        #     (before ";")     (after ";")
        
        # x goes through convolutions
        h = self.conv_in(x)
        
        # sigma_embedding modifies the processing
        # at every layer via adaptive normalization
        for block in self.blocks:
            h = block(h, sigma_embedding)  # σ affects how features are normalized
        
        return self.conv_out(h)
```

**Why this matters for EDM specifically:** In the expression $F_\theta\!\bigl(c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n});\; c_{\text{noise}}(\sigma)\bigr)$:

- Argument 1, $c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n})$, is a **3D image tensor** (channels × height × width) — the normalized noisy input that flows through convolutions
- Argument 2, $c_{\text{noise}}(\sigma) = \frac{1}{4}\ln(\sigma)$, is a **single scalar** — noise level metadata that gets embedded and injected into every layer to tell the network "you're currently working at noise level $\sigma$"

The semicolon makes this asymmetry explicit in the math.

### 0.11 — Bold vs. Non-Bold Letters

| Style | Meaning | Example |
|---|---|---|
| $\mathbf{y}$ (bold) | A **vector** (an entire image with thousands of pixels) | $\mathbf{y} \in \mathbb{R}^{3072}$ for a 32×32×3 image |
| $\sigma$ (non-bold) | A **scalar** (a single number) | $\sigma = 0.5$ |
| $\mathbf{I}$ (bold) | A **matrix** (the identity matrix) | $\mathbf{I} \in \mathbb{R}^{d \times d}$ |

### 0.12 — Other Notation Used Later

| Symbol | Meaning |
|---|---|
| $c_{\text{skip}}(\sigma)$ | A scalar function of $\sigma$ — weight for the skip connection. Not learned; it's a fixed formula. |
| $c_{\text{out}}(\sigma)$ | A scalar function of $\sigma$ — scaling factor for the network output |
| $c_{\text{in}}(\sigma)$ | A scalar function of $\sigma$ — scaling factor for the network input |
| $c_{\text{noise}}(\sigma)$ | A scalar function of $\sigma$ — transforms $\sigma$ into a conditioning input (EDM uses $\frac{1}{4}\ln\sigma$) |
| $F_{\text{target}}$ | The "answer key" — what the network *should* output for a given ($\mathbf{y}$, $\mathbf{n}$, $\sigma$) |
| $\frac{d}{d\,c_{\text{skip}}}$ | Derivative with respect to $c_{\text{skip}}$ — used to find the minimum of a function |
| $\implies$ | "implies" or "therefore" — means the left side logically leads to the right side |
| $\approx$ | "approximately equal to" — used when a quantity is close but not exact |
| $\to$ | "approaches" or "tends to" — e.g., $c_{\text{skip}} \to 1$ means $c_{\text{skip}}$ gets close to 1 |
| $\ll$, $\gg$ | "much less than", "much greater than" — e.g., $\sigma \ll \sigma_{\text{data}}$ means $\sigma$ is far smaller |
| $\sum_{i=1}^{d}$ | "sum from $i=1$ to $d$" — add up $d$ terms |
| $\ln(\cdot)$ | Natural logarithm — the inverse of $e^x$. Compresses large ranges (e.g., $\ln(80) \approx 4.4$) |
| $\sqrt{\cdot}$ | Square root — $\sqrt{x}$ is the number that, when squared, gives $x$ |
| $\boxed{\cdots}$ | A boxed equation is a **final result** — a derived formula ready to use |

---

## 1. Starting Point: The Denoising Score Matching Loss

The fundamental training objective in diffusion models is **denoising score matching** (Eq. 2 in the paper). For a given noise level $\sigma$, we want a denoiser $D_\theta$ that minimizes:

$$L(D_\theta;\, \sigma) \;=\; \mathbb{E}_{\mathbf{y}\sim p_{\text{data}}}\;\mathbb{E}_{\mathbf{n}\sim\mathcal{N}(\mathbf{0},\,\sigma^2\mathbf{I})} \left[\, \bigl\| D_\theta(\mathbf{y}+\mathbf{n};\,\sigma) - \mathbf{y} \bigr\|_2^2 \,\right]$$

**Reading this equation out loud, piece by piece:**

> *"The loss $L$ of our model $D_\theta$ at noise level $\sigma$ equals: pick a random clean image $\mathbf{y}$ from the training data, pick a random noise vector $\mathbf{n}$ from a Gaussian with standard deviation $\sigma$, add them to get a noisy image $\mathbf{y}+\mathbf{n}$, feed it to our denoiser $D_\theta$ along with the noise level $\sigma$, compute how far the output is from the true clean image $\mathbf{y}$ (sum of squared pixel differences), and average this error over all possible images and noise vectors."*

**Annotated component breakdown:**

$$L(\underbrace{D_\theta}_{\text{our model}};\, \underbrace{\sigma}_{\text{fixed noise level}}) \;=\; \underbrace{\mathbb{E}_{\mathbf{y}\sim p_{\text{data}}}}_{\text{average over all images}}\;\underbrace{\mathbb{E}_{\mathbf{n}\sim\mathcal{N}(\mathbf{0},\,\sigma^2\mathbf{I})}}_{\text{average over all noise vectors}} \left[\, \bigl\| \underbrace{D_\theta(\mathbf{y}+\mathbf{n};\,\sigma)}_{\text{model's prediction}} - \underbrace{\mathbf{y}}_{\text{ground truth}} \bigr\|_2^2 \,\right]$$

**What this looks like in Python pseudocode:**

```python
# What the math says (conceptual — can't actually run this)
def loss_at_sigma(model, sigma, entire_dataset, infinite_noise_samples):
    total = 0
    count = 0
    for y in entire_dataset:                    # E over y ~ p_data
        for n in infinite_noise_samples:        # E over n ~ N(0, σ²I)
            noisy_input = y + n                 # y + n
            prediction = model(noisy_input, sigma)  # D_θ(y+n; σ)
            error = sum((prediction - y)**2)    # ‖...‖₂²
            total += error
            count += 1
    return total / count                        # the E (average)

# What actually happens each training step (the approximation)
def training_step(model, sigma):
    y = random_image_from_dataset()             # one sample ≈ E_y
    n = random_normal(mean=0, std=sigma)        # one sample ≈ E_n
    prediction = model(y + n, sigma)
    loss = sum((prediction - y)**2)
    loss.backward()                             # update θ
```

Here:

- $\mathbf{y}$ is a clean training image drawn from $p_{\text{data}}$ — one specific photo of a cat, car, etc.
- $\mathbf{n}$ is Gaussian noise with standard deviation $\sigma$ — a random "static" pattern added to the image
- $\mathbf{y} + \mathbf{n}$ is the noisy image that the denoiser receives — the clean image buried under noise
- $D_\theta(\mathbf{y}+\mathbf{n};\,\sigma)$ is the denoiser's attempt to recover the clean image from the noisy input. The $\theta$ reminds us this is our trained model (not a perfect oracle)
- $\|\cdots\|_2^2$ measures how many pixels the denoiser got wrong, and by how much

The overall training loss integrates over **all noise levels** with a weighting function (Eq. 108 in Appendix B.6):

$$L(D_\theta) \;=\; \mathbb{E}_{\sigma,\,\mathbf{y},\,\mathbf{n}} \left[\, \lambda(\sigma)\,\bigl\| D_\theta(\mathbf{y}+\mathbf{n};\,\sigma) - \mathbf{y} \bigr\|_2^2 \,\right]$$

**Reading this equation:**

> *"The total loss of our model equals: pick a random noise level $\sigma$ (from the training distribution $p_{\text{train}}$), pick a random clean image $\mathbf{y}$, pick random noise $\mathbf{n}$, compute the denoising error, multiply it by a weight $\lambda(\sigma)$ that controls how much we care about this noise level, and average everything."*

**Annotated component breakdown:**

$$L(\underbrace{D_\theta}_{\text{our model}}) \;=\; \underbrace{\mathbb{E}_{\sigma,\,\mathbf{y},\,\mathbf{n}}}_{\text{average over ALL three random variables}} \left[\, \underbrace{\lambda(\sigma)}_{\text{importance weight}}\,\bigl\| \underbrace{D_\theta(\mathbf{y}+\mathbf{n};\,\sigma)}_{\text{model's prediction}} - \underbrace{\mathbf{y}}_{\text{ground truth}} \bigr\|_2^2 \,\right]$$

**Key difference from the first equation:** The first equation ($L(D_\theta;\sigma)$) fixes $\sigma$ and averages over images and noise. This equation ($L(D_\theta)$) *also* averages over $\sigma$ — it's the total loss across all noise levels. The $\lambda(\sigma)$ multiplier is new: it says "when $\sigma$ happens to be large, weight this error by $\lambda(\text{large})$; when $\sigma$ is small, weight by $\lambda(\text{small})$."

**What this looks like in Python pseudocode:**

```python
# What the math says (conceptual)
def total_loss(model):
    total = 0
    count = 0
    for sigma in all_noise_levels:          # E over σ ~ p_train
        weight = lambda_fn(sigma)           # λ(σ)
        for y in entire_dataset:            # E over y
            for n in all_noise_vectors:     # E over n
                prediction = model(y + n, sigma)
                error = sum((prediction - y)**2)
                total += weight * error     # λ(σ) · ‖...‖²
                count += 1
    return total / count

# What actually happens each training step
def training_step(model):
    sigma = sample_from_log_normal()        # one σ ≈ E_σ
    y = random_image_from_dataset()         # one image ≈ E_y
    n = random_normal(mean=0, std=sigma)    # one noise ≈ E_n
    weight = lambda_fn(sigma)               # λ(σ)
    prediction = model(y + n, sigma)
    loss = weight * sum((prediction - y)**2)
    loss.backward()
```

The compact subscript $\mathbb{E}_{\sigma,\mathbf{y},\mathbf{n}}$ is shorthand for the three nested expectations — we are averaging over all three random variables simultaneously.

**Why $\lambda(\sigma)$?** Not all noise levels are equally important. The weight $\lambda(\sigma)$ lets us say "care more about medium noise levels" (where the model can actually learn useful structure) and "care less about extreme noise levels" (where the task is either trivially easy or impossibly hard).

where $\sigma \sim p_{\text{train}}$ and $\lambda(\sigma)$ is a per-noise-level weight.

---

## 2. The Preconditioning Structure (Equation 7)

Training $D_\theta$ as a raw neural network would be problematic — the input magnitude $\|\mathbf{y}+\mathbf{n}\|$ varies enormously with $\sigma$. The EDM paper wraps the raw network $F_\theta$ in a preconditioning shell:

$$D_\theta(\mathbf{x};\,\sigma) \;=\; c_{\text{skip}}(\sigma)\,\mathbf{x} \;+\; c_{\text{out}}(\sigma)\, F_\theta\!\bigl(c_{\text{in}}(\sigma)\,\mathbf{x};\; c_{\text{noise}}(\sigma)\bigr)$$

**Reading this equation piece by piece:**

> *"The denoiser $D_\theta$ applied to noisy image $\mathbf{x}$ at noise level $\sigma$ equals: take the input image $\mathbf{x}$ and multiply it by the skip weight $c_{\text{skip}}(\sigma)$ (this is the 'shortcut' path), then separately feed a scaled version $c_{\text{in}}(\sigma)\,\mathbf{x}$ of the input into the raw neural network $F_\theta$ (with noise conditioning $c_{\text{noise}}(\sigma)$), scale the network's output by $c_{\text{out}}(\sigma)$, and add the two paths together."*

**Why is $\mathbf{x}$ multiplied by scalars like $c_{\text{skip}}(\sigma)$?** Each $c$ function outputs a single number (a scalar) that depends on the noise level $\sigma$. When you multiply a scalar by a vector (image), every pixel gets multiplied by that same number. So $c_{\text{skip}}(\sigma)\,\mathbf{x}$ means "dim or brighten the entire image by factor $c_{\text{skip}}$." These are **not** learned — they are fixed formulas computed from $\sigma$.

Each component has a specific role:

| Component | Role |
|---|---|
| $c_{\text{skip}}(\sigma)\,\mathbf{x}$ | **Skip connection** — passes the input through directly, weighted by $c_{\text{skip}}$. At low noise, $c_{\text{skip}} \approx 1$ (trust the input). At high noise, $c_{\text{skip}} \approx 0$ (input is garbage, don't trust it). |
| $c_{\text{in}}(\sigma)$ | **Input scaling** — normalizes the network input to unit variance, so $F_\theta$ always sees inputs of similar magnitude regardless of $\sigma$ |
| $c_{\text{out}}(\sigma)$ | **Output scaling** — scales the network output to the correct magnitude. Kept small to avoid amplifying network errors |
| $c_{\text{noise}}(\sigma)$ | **Noise conditioning** — encodes $\sigma$ as a conditioning signal for $F_\theta$. EDM uses $\frac{1}{4}\ln(\sigma)$, which compresses the huge range $[0.002, 80]$ into a manageable range for the network |

---

## 3. The Algebraic Derivation — From $D_\theta$ Loss to $F_\theta$ Loss

We now substitute the preconditioning formula (Eq. 7) into the denoising loss. Let $\mathbf{x} = \mathbf{y} + \mathbf{n}$ (the noisy image).

The goal: We want to rewrite the loss so that instead of being about $D_\theta$ (the full denoiser), it's about $F_\theta$ (the raw network). This will reveal what $F_\theta$ actually needs to learn.

### Step 3.1 — Expand $D_\theta$

Replace $D_\theta(\mathbf{x};\,\sigma)$ with its definition from Eq. 7:

$$\bigl\| D_\theta(\mathbf{x};\,\sigma) - \mathbf{y} \bigr\|_2^2 \;=\; \Bigl\| \bigl[c_{\text{skip}}(\sigma)\,\mathbf{x} + c_{\text{out}}(\sigma)\,F_\theta(c_{\text{in}}(\sigma)\,\mathbf{x};\,c_{\text{noise}}(\sigma))\bigr] - \mathbf{y} \Bigr\|_2^2$$

> *"We plugged in the definition. Inside the norm, we now have: (skip path + network path) minus the true clean image."*

This is Equation 109 in Appendix B.6.

### Step 3.2 — Rearrange to Isolate $F_\theta$

Group the $F_\theta$ term on one side and everything else on the other. Since $\|\mathbf{a} - \mathbf{b}\|^2 = \|\mathbf{b} - \mathbf{a}\|^2$ (squaring removes sign), we can rearrange freely:

$$= \Bigl\| c_{\text{out}}(\sigma)\,F_\theta(c_{\text{in}}(\sigma)\,\mathbf{x};\,c_{\text{noise}}(\sigma)) - \bigl[\mathbf{y} - c_{\text{skip}}(\sigma)\,\mathbf{x}\bigr] \Bigr\|_2^2$$

> *"We moved $c_{\text{skip}}(\sigma)\,\mathbf{x}$ from being added on the left to being subtracted on the right. Now one side has the network output (scaled by $c_{\text{out}}$) and the other side has 'what the network output should match'."*

This is Equation 110 — we simply moved $c_{\text{skip}}(\sigma)\,\mathbf{x}$ from the left part to the right part.

### Step 3.3 — Factor Out $c_{\text{out}}$

This is the key algebraic trick. Since $c_{\text{out}}$ is a scalar (just a number), and we know that $\|a \cdot \mathbf{v}\|_2^2 = a^2 \|\mathbf{v}\|_2^2$ (scaling a vector by $a$ scales its squared norm by $a^2$), we can factor $c_{\text{out}}(\sigma)$ out of the norm:

$$= c_{\text{out}}(\sigma)^2 \left\| F_\theta\!\bigl(c_{\text{in}}(\sigma)\,\mathbf{x};\,c_{\text{noise}}(\sigma)\bigr) - \frac{1}{c_{\text{out}}(\sigma)}\bigl(\mathbf{y} - c_{\text{skip}}(\sigma)\,\mathbf{x}\bigr) \right\|_2^2$$

> *"We divided everything inside the norm by $c_{\text{out}}$ and compensated by multiplying outside by $c_{\text{out}}^2$. Now the left term inside the norm is the raw network output $F_\theta(\cdots)$ (no scaling!), and the right term is the effective target — what $F_\theta$ should try to output."*

This is Equation 111. **This step is why the derivation matters** — it reveals the **effective training target** of $F_\theta$: the expression $\frac{1}{c_{\text{out}}(\sigma)}(\mathbf{y} - c_{\text{skip}}(\sigma)\,\mathbf{x})$.

### Step 3.4 — Substitute $\mathbf{x} = \mathbf{y} + \mathbf{n}$ and Include $\lambda(\sigma)$

Multiplying by $\lambda(\sigma)$ and taking the expectation:

$$L(D_\theta) = \mathbb{E}_{\sigma,\,\mathbf{y},\,\mathbf{n}} \left[\; \underbrace{\lambda(\sigma)\,c_{\text{out}}(\sigma)^2}_{\text{effective weight}}\; \left\| \underbrace{F_\theta\!\bigl(c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n});\; c_{\text{noise}}(\sigma)\bigr)}_{\text{network output}} - \underbrace{\frac{1}{c_{\text{out}}(\sigma)}\bigl(\mathbf{y} - c_{\text{skip}}(\sigma)\cdot(\mathbf{y}+\mathbf{n})\bigr)}_{\text{effective training target}} \right\|_2^2 \;\right]$$

> *"The total loss equals: average over all noise levels, images, and noise vectors of [effective weight × (what the network actually outputs − what it should output)$^2$]."*

**This is Equation 8** (= Equation 112 in the appendix). The image shows the inner squared-norm term.

**What Equation 8 looks like in Python pseudocode:**

```python
def edm_training_step(F_theta, sigma, sigma_data):
    y = random_image_from_dataset()
    n = random_normal(mean=0, std=sigma, shape=y.shape)

    # Preconditioning scalings (fixed formulas, NOT learned)
    totvar  = sigma**2 + sigma_data**2
    c_skip  = sigma_data**2 / totvar
    c_out   = sigma * sigma_data / sqrt(totvar)
    c_in    = 1.0 / sqrt(totvar)
    c_noise = 0.25 * log(sigma)

    # What F_θ receives and what it should output
    network_input  = c_in * (y + n)                     # normalized to ~unit variance
    network_output = F_theta(network_input, c_noise)     # the raw network
    effective_target = (1/c_out) * (y - c_skip * (y+n))  # the "answer key"

    # The loss (this is the image equation!)
    lambda_weight = 1.0 / c_out**2
    loss = lambda_weight * c_out**2 * sum((network_output - effective_target)**2)
    #    = sum((network_output - effective_target)**2)  ← weights cancel to 1!
    loss.backward()
```

**Why this matters:** Before this derivation, the loss was written in terms of $D_\theta$ — the combined system. After this derivation, we can see exactly what the raw network $F_\theta$ needs to learn (the effective target) and how its errors are weighted (the effective weight $\lambda \cdot c_{\text{out}}^2$). This is what lets us design $c_{\text{skip}}$, $c_{\text{out}}$, $c_{\text{in}}$, and $\lambda$ from principled requirements.

---

## 4. Understanding the Effective Training Target

The effective training target that $F_\theta$ must learn to predict is:

$$F_{\text{target}}(\mathbf{y},\,\mathbf{n};\,\sigma) \;=\; \frac{1}{c_{\text{out}}(\sigma)}\,\bigl(\mathbf{y} - c_{\text{skip}}(\sigma)\cdot(\mathbf{y}+\mathbf{n})\bigr)$$

> *"$F_{\text{target}}$ is a function of the clean image $\mathbf{y}$, the noise $\mathbf{n}$, and the noise level $\sigma$. It equals: take the clean image, subtract the skip-connection's contribution (which is just $c_{\text{skip}}$ times the noisy input), then divide everything by $c_{\text{out}}$."*

**Note:** $F_{\text{target}}$ is NOT something the network computes — it's the **answer key**. During training, for each ($\mathbf{y}$, $\mathbf{n}$, $\sigma$) triple, we know what $F_\theta$ *should* output, and $F_{\text{target}}$ is that ideal output.

Let us expand it by distributing $c_{\text{skip}}$ across $(\mathbf{y}+\mathbf{n})$:

$$F_{\text{target}} = \frac{1}{c_{\text{out}}(\sigma)}\,\Bigl[\underbrace{\bigl(1 - c_{\text{skip}}(\sigma)\bigr)\,\mathbf{y}}_{\text{clean image part}} \;-\; \underbrace{c_{\text{skip}}(\sigma)\,\mathbf{n}}_{\text{noise part}}\Bigr]$$

> *"The target is a weighted combination: some fraction of the clean image minus some fraction of the noise, all divided by $c_{\text{out}}$ to normalize to unit variance."*

This reveals the key insight: **the network's training target is a $\sigma$-dependent mixture of the clean image and the noise**, both rescaled to unit variance by $c_{\text{out}}$.

### Behavior at Different Noise Levels

Using the EDM formulas $c_{\text{skip}} = \sigma_{\text{data}}^2/(\sigma^2+\sigma_{\text{data}}^2)$ and $c_{\text{out}} = \sigma\,\sigma_{\text{data}}/\sqrt{\sigma^2+\sigma_{\text{data}}^2}$:

**Low noise** ($\sigma \ll \sigma_{\text{data}}$): $\;c_{\text{skip}} \to 1$, so:

$$F_{\text{target}} \;\approx\; \frac{-c_{\text{skip}}}{c_{\text{out}}}\,\mathbf{n} \;\approx\; \frac{-\mathbf{n}}{c_{\text{out}}}$$

The network learns to predict (scaled) noise — similar to $\epsilon$-prediction in DDPM.

**High noise** ($\sigma \gg \sigma_{\text{data}}$): $\;c_{\text{skip}} \to 0$, so:

$$F_{\text{target}} \;\approx\; \frac{\mathbf{y}}{c_{\text{out}}}$$

The network learns to predict the (scaled) clean image directly — similar to $\mathbf{x}_0$-prediction.

**Intermediate noise** ($\sigma \approx \sigma_{\text{data}}$): The target is a balanced mix of both signal and noise — similar in spirit to $\mathbf{v}$-prediction, but derived from optimality principles rather than chosen heuristically.

---

## 5. Deriving the Specific Formulas for $c_{\text{skip}}$, $c_{\text{out}}$, $c_{\text{in}}$

The paper derives these from three requirements (Appendix B.6, Eqs. 114–144). Each requirement is a simple, intuitive "wish" about how training should behave, and the math gives us the formula that satisfies it.

### 5.1 — Require Unit Variance Input (gives $c_{\text{in}}$)

**The wish:** "No matter what the noise level $\sigma$ is, the network input should have unit variance."

**Why?** Neural networks train best when their inputs are normalized. If the input magnitude varies from 0.002 to 80 depending on $\sigma$, the network's weights can't adapt to all scales simultaneously.

**The math:** The network receives $c_{\text{in}}(\sigma) \cdot (\mathbf{y}+\mathbf{n})$. We require:

$$\text{Var}_{\mathbf{y},\mathbf{n}}\!\bigl[c_{\text{in}}(\sigma)\cdot(\mathbf{y}+\mathbf{n})\bigr] = 1$$

> *"The variance of the scaled noisy input (averaged over all possible images and noise) should equal 1."*

Since $\mathbf{y}$ and $\mathbf{n}$ are independent with variances $\sigma_{\text{data}}^2$ and $\sigma^2$, and $c_{\text{in}}$ is a scalar constant (for a given $\sigma$):

$$c_{\text{in}}(\sigma)^2 \cdot \underbrace{(\sigma_{\text{data}}^2 + \sigma^2)}_{\text{Var}[\mathbf{y}+\mathbf{n}]} = 1$$

> *"$c_{\text{in}}$ squared times the total variance of the noisy input must equal 1."*

Solving for $c_{\text{in}}$:

$$\boxed{c_{\text{in}}(\sigma) = \frac{1}{\sqrt{\sigma^2 + \sigma_{\text{data}}^2}}}$$

**Sanity check:** At $\sigma = 80$ (heavy noise), $c_{\text{in}} \approx 1/80 = 0.0125$, so the huge noisy input gets shrunk down. At $\sigma = 0.002$ (tiny noise), $c_{\text{in}} \approx 1/0.5 = 2$, so the clean-ish input gets scaled up slightly. Either way, the network sees something of magnitude $\approx 1$.

### 5.2 — Require Unit Variance Target AND Minimum $c_{\text{out}}$ (gives $c_{\text{skip}}$ and $c_{\text{out}}$)

**The wishes:**
1. "The training target $F_{\text{target}}$ should also have unit variance" (so the network output is well-conditioned)
2. "Among all possible $c_{\text{skip}}$ that achieve this, pick the one that makes $c_{\text{out}}$ as small as possible" (to minimize how much network errors get amplified)

**Step A — Unit variance target:**

First, compute the variance of the effective target from Section 4:

$$\text{Var}_{\mathbf{y},\mathbf{n}}\!\bigl[F_{\text{target}}\bigr] = \frac{1}{c_{\text{out}}^2}\left[\underbrace{(1-c_{\text{skip}})^2\,\sigma_{\text{data}}^2}_{\text{variance from }\mathbf{y}\text{ part}} + \underbrace{c_{\text{skip}}^2\,\sigma^2}_{\text{variance from }\mathbf{n}\text{ part}}\right] \;=\; 1$$

> *"The variance of the target has two contributions: one from the clean image part and one from the noise part (they add because $\mathbf{y}$ and $\mathbf{n}$ are independent). The whole thing divided by $c_{\text{out}}^2$ must equal 1."*

This gives us a relationship between $c_{\text{out}}$ and $c_{\text{skip}}$:

$$c_{\text{out}}^2 = (1-c_{\text{skip}})^2\,\sigma_{\text{data}}^2 + c_{\text{skip}}^2\,\sigma^2$$

**Step B — Minimize $c_{\text{out}}$ via $c_{\text{skip}}$:**

Now choose $c_{\text{skip}}$ to **minimize** $c_{\text{out}}$ (minimize error amplification). To find the minimum of a function, we take its derivative and set it to zero:

$$\frac{d\,c_{\text{out}}^2}{d\,c_{\text{skip}}} = 0$$

> *"Find the value of $c_{\text{skip}}$ where $c_{\text{out}}^2$ stops decreasing and starts increasing — that's the minimum."*

Computing the derivative (using the chain rule on each term):

$$2\sigma_{\text{data}}^2(c_{\text{skip}}-1) + 2\sigma^2\,c_{\text{skip}} = 0$$

$$(\sigma^2+\sigma_{\text{data}}^2)\,c_{\text{skip}} = \sigma_{\text{data}}^2$$

$$\boxed{c_{\text{skip}}(\sigma) = \frac{\sigma_{\text{data}}^2}{\sigma^2 + \sigma_{\text{data}}^2}}$$

**Sanity check:** At $\sigma = 0$ (no noise), $c_{\text{skip}} = 1$ — the skip connection passes input through unchanged (because the input IS the clean image). At $\sigma = 80$ (extreme noise), $c_{\text{skip}} \approx 0$ — the skip connection is turned off (because the input is pure garbage).

**Step C — Compute $c_{\text{out}}$:**

Substituting the optimal $c_{\text{skip}}$ back into the $c_{\text{out}}^2$ formula:

$$c_{\text{out}}^2 = \left(\frac{\sigma^2}{\sigma^2+\sigma_{\text{data}}^2}\right)^2 \sigma_{\text{data}}^2 + \left(\frac{\sigma_{\text{data}}^2}{\sigma^2+\sigma_{\text{data}}^2}\right)^2 \sigma^2 = \frac{(\sigma\,\sigma_{\text{data}})^2}{\sigma^2+\sigma_{\text{data}}^2}$$

$$\boxed{c_{\text{out}}(\sigma) = \frac{\sigma\;\sigma_{\text{data}}}{\sqrt{\sigma^2 + \sigma_{\text{data}}^2}}}$$

**Sanity check:** $c_{\text{out}}$ is always $\leq \sigma_{\text{data}} = 0.5$. Compare this with prior methods where $c_{\text{out}} = \sigma$, which could be as large as 80 — meaning any tiny error by $F_\theta$ would get amplified 80×! EDM's formulation caps this amplification at 0.5×.

### 5.3 — Require Uniform Effective Weight (gives $\lambda$)

**The wish:** "Every noise level should contribute equally to the total loss at initialization."

From Equation 8, the effective per-sample weight is $\lambda(\sigma)\,c_{\text{out}}(\sigma)^2$. Setting this to 1 (uniform across all $\sigma$):

$$\lambda(\sigma)\,c_{\text{out}}(\sigma)^2 = 1 \quad\implies\quad \lambda(\sigma) = \frac{1}{c_{\text{out}}(\sigma)^2}$$

Substituting our formula for $c_{\text{out}}$:

$$\lambda(\sigma) = \frac{1}{c_{\text{out}}(\sigma)^2} = \frac{\sigma^2+\sigma_{\text{data}}^2}{(\sigma\;\sigma_{\text{data}})^2}$$

> *"At noise levels where $c_{\text{out}}$ is small (meaning the network's contribution is small), we weight the loss more heavily to compensate. At noise levels where $c_{\text{out}}$ is large, we weight less."*

---

## 6. Verification: Initial Loss Equals 1

This section shows a beautiful sanity check: if you plug in all the derived formulas and compute the loss *before any training happens*, you get exactly 1 at every noise level. This confirms the preconditioning is perfectly balanced from the start.

**Setup:** The paper initializes the output layer of $F_\theta$ to zero. At initialization, $F_\theta(\cdot) = \mathbf{0}$ (the network outputs a blank image), so the denoiser reduces to just the skip connection:

$$D_\theta(\mathbf{x};\sigma) = c_{\text{skip}}(\sigma)\,\mathbf{x} + c_{\text{out}}(\sigma)\cdot\underbrace{\mathbf{0}}_{F_\theta = 0} = c_{\text{skip}}(\sigma)\,(\mathbf{y}+\mathbf{n})$$

> *"Before training, the model's best guess is just the noisy input scaled by $c_{\text{skip}}$. At low noise this is almost the clean image (good guess!). At high noise this is almost zero (a safe default)."*

**Computing the initial loss per noise level** (shown in Eqs. 145–151 of the paper). We substitute $D_\theta = c_{\text{skip}}(\mathbf{y}+\mathbf{n})$ into the loss:

$$\lambda(\sigma)\,\mathbb{E}\!\left[\left\|c_{\text{skip}}(\sigma)(\mathbf{y}+\mathbf{n}) - \mathbf{y}\right\|^2\right]$$

> *"Error = (skip-scaled noisy image) minus (true clean image), squared and averaged."*

Now substitute the formulas $\lambda(\sigma) = \frac{\sigma^2+\sigma_{\text{data}}^2}{(\sigma\,\sigma_{\text{data}})^2}$ and $c_{\text{skip}}(\sigma) = \frac{\sigma_{\text{data}}^2}{\sigma^2+\sigma_{\text{data}}^2}$:

$$= \frac{\sigma^2+\sigma_{\text{data}}^2}{(\sigma\,\sigma_{\text{data}})^2} \;\mathbb{E}\!\left[\left\|\frac{\sigma_{\text{data}}^2}{\sigma^2+\sigma_{\text{data}}^2}(\mathbf{y}+\mathbf{n}) - \mathbf{y}\right\|^2\right]$$

**Simplifying the inside of the norm** — distribute $c_{\text{skip}}$ across $(\mathbf{y}+\mathbf{n})$ and collect terms involving $\mathbf{y}$ and $\mathbf{n}$:

$$\frac{\sigma_{\text{data}}^2}{\sigma^2+\sigma_{\text{data}}^2}\mathbf{y} + \frac{\sigma_{\text{data}}^2}{\sigma^2+\sigma_{\text{data}}^2}\mathbf{n} - \mathbf{y} = -\frac{\sigma^2}{\sigma^2+\sigma_{\text{data}}^2}\mathbf{y} + \frac{\sigma_{\text{data}}^2}{\sigma^2+\sigma_{\text{data}}^2}\mathbf{n}$$

> *"The $\mathbf{y}$ coefficient became $\frac{\sigma_{\text{data}}^2}{\sigma^2+\sigma_{\text{data}}^2} - 1 = -\frac{\sigma^2}{\sigma^2+\sigma_{\text{data}}^2}$."*

After factoring out $\frac{1}{\sigma^2+\sigma_{\text{data}}^2}$ from both terms:

$$= \frac{1}{\sigma^2+\sigma_{\text{data}}^2}\;\mathbb{E}\!\left[\left\|\frac{\sigma_{\text{data}}}{\sigma}\,\mathbf{n} - \frac{\sigma}{\sigma_{\text{data}}}\,\mathbf{y}\right\|^2\right]$$

**Now we use two key facts:**

1. **"Zero-mean":** Both $\mathbf{y}$ and $\mathbf{n}$ have mean zero (on average, across the whole dataset, pixel values center around zero; noise is zero-mean by construction). This means the cross-term $\mathbb{E}[\mathbf{y} \cdot \mathbf{n}] = 0$ — on average, the image and noise don't "align" in any direction. So when we expand $\|\mathbf{a} - \mathbf{b}\|^2 = \|\mathbf{a}\|^2 - 2\mathbf{a}\cdot\mathbf{b} + \|\mathbf{b}\|^2$, the middle cross-term vanishes.

2. **"Independent":** The noise $\mathbf{n}$ is generated without any knowledge of the image $\mathbf{y}$, so knowing $\mathbf{y}$ tells you nothing about $\mathbf{n}$ and vice versa. This is why $\text{Var}[\mathbf{y}+\mathbf{n}] = \text{Var}[\mathbf{y}] + \text{Var}[\mathbf{n}]$ — variances add for independent variables.

Applying these facts:

$$= \frac{1}{\sigma^2+\sigma_{\text{data}}^2}\left(\underbrace{\frac{\sigma_{\text{data}}^2}{\sigma^2}\,\text{Var}(\mathbf{n})}_{\text{noise contribution}} + \underbrace{\frac{\sigma^2}{\sigma_{\text{data}}^2}\,\text{Var}(\mathbf{y})}_{\text{image contribution}}\right)$$

> *"Each term contributes: (coefficient²) × (variance of the variable)."*

Substituting $\text{Var}(\mathbf{n}) = \sigma^2$ and $\text{Var}(\mathbf{y}) = \sigma_{\text{data}}^2$:

$$= \frac{1}{\sigma^2+\sigma_{\text{data}}^2}\left(\frac{\sigma_{\text{data}}^2}{\sigma^2}\cdot\sigma^2 + \frac{\sigma^2}{\sigma_{\text{data}}^2}\cdot\sigma_{\text{data}}^2\right) = \frac{1}{\sigma^2+\sigma_{\text{data}}^2}\left(\sigma_{\text{data}}^2 + \sigma^2\right) = 1$$

> *"Everything cancels perfectly! The numerator and denominator are the same."*

The initial loss is **exactly 1** at every noise level — confirming the preconditioning is well-balanced from the start of training. This is shown as the green curve in Figure 5a of the paper.

**Why this matters practically:** If the initial loss were 100 at some noise levels and 0.001 at others, the gradient signal would be dominated by the high-loss noise levels, and the network would learn unevenly. By starting at a uniform loss of 1 everywhere, every noise level gets equal attention from the first training step.

---

## 7. Summary: Why This Formulation Matters

The transformation from the $D_\theta$ loss to the $F_\theta$ loss (Equation 8) is not just algebraic convenience. It achieves three practical goals simultaneously:

1. **Stable input magnitude** ($c_{\text{in}}$): The network always sees unit-variance inputs, preventing saturation or vanishing activations.

2. **Bounded error amplification** ($c_{\text{skip}}$ minimizes $c_{\text{out}}$): Unlike prior methods where $c_{\text{out}} = \sigma$ (amplifying errors up to 80×), EDM's $c_{\text{out}}$ is bounded by $\sigma_{\text{data}}$ (typically 0.5).

3. **Uniform loss landscape** ($\lambda$ and unit-variance target): The network faces equally difficult tasks at all noise levels, preventing training from being dominated by any particular $\sigma$ range.

The result, as shown in Table 2 of the paper, is that preconditioning alone (config D) does not dramatically change FID, but it **stabilizes training enough** to enable the more impactful loss function redesign (config E) that produces the major quality improvements.

In [ ]:
def scalings(sig):
    """
    Compute Karras pre-conditioning scalings.
    
    These formulas come from the EDM paper and ensure:
    - Network input has unit variance
    - Network output has unit variance
    - Skip connection is properly weighted
    
    Args:
        sig: Noise level (sigma), can be scalar or tensor
    
    Returns:
        c_skip: Skip connection weight
        c_out: Output scaling
        c_in: Input scaling
    """
    # Total variance = noise variance + data variance
    totvar = sig**2 + sig_data**2
    
    # c_skip: Weight for skip connection (passes input through)
    # At high noise (sig >> sig_data): c_skip -> 0 (don't trust input)
    # At low noise (sig << sig_data): c_skip -> 1 (trust input)
    c_skip = sig_data**2 / totvar
    
    # c_out: Scaling for network output
    # Ensures output contribution has correct magnitude
    c_out = sig * sig_data / totvar.sqrt()
    
    # c_in: Scaling for network input
    # Normalizes input to have approximately unit variance
    c_in = 1 / totvar.sqrt()
    
    return c_skip, c_out, c_in

![](img_4.png)

Here distribution of sigma as shown in the image above is a log normal curve.

**Understanding the scalings:**

The pre-conditioned model output is:
```
D(x, sigma) = c_skip * x + c_out * F(c_in * x, sigma)
```

Where:
- `x` is the noisy input
- `F` is the neural network
- `D` is the denoised output

```
At HIGH noise (sigma >> sig_data):
- c_skip ≈ 0 (ignore input, it's mostly noise)
- c_out ≈ sig_data (network fully responsible)
- c_in ≈ 1/sigma (scale down noisy input)

At LOW noise (sigma << sig_data):
- c_skip ≈ 1 (trust input, it's mostly signal)
- c_out ≈ sigma/sig_data (network makes small corrections)
- c_in ≈ 1/sig_data (scale by data std)
```

### 🎮 Interactive: The Three Dials — c_skip, c_out, c_in

The three formulas in `scalings()` are one line each, but their *shapes* as functions of σ are what make them click.

- The **pipeline diagram** at the top is live: the teal **skip highway's thickness is c_skip**, and each ×-gate is sized
  by its dial. Click any box or gate to interrogate it with real numbers at the current σ.
- **Drag the ✏️ marker** across the dial-curves plot (or press a stage chip — each one replays a σ-sweep) and watch the
  matching `scalings()` line light up while the others stay visible.
- Move **σ_data** and notice every crossover tracks σ ≈ σ_data: the dials are calibrated *relative to the data's typical
  size* — exactly why `sig_data` exists.
- The bars on the right are the punchline: whatever σ you choose, the network's **input variance and target variance are
  pinned at 1.00**. That is the entire purpose of these formulas.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_scalings_explorer.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/karras_scalings_explorer.html", height="950px")

---
## Log-Normal Sigma Sampling

The EDM paper recommends sampling sigma from a **log-normal distribution** during training, rather than uniform. This concentrates samples around the most important noise levels.

In [ ]:
# Sample sigmas from log-normal distribution
# log(sigma) ~ N(-1.2, 1.2^2)
# This gives sigma values mostly between 0.01 and 10
sig_samp = (torch.randn([10000]) * 1.2 - 1.2).exp()

**How it works:**

```
1. Sample z ~ N(0, 1)           # Standard normal
2. Compute log(sigma) = z * 1.2 - 1.2  # Shift and scale
3. Compute sigma = exp(log(sigma))     # Exponentiate

Result:
- Mean of log(sigma) = -1.2  →  Median sigma ≈ 0.3
- Std of log(sigma) = 1.2    →  Wide spread on log scale
```

In [ ]:
# Visualize the distribution (histogram)
plt.hist(sig_samp, bins=50)
plt.xlabel('Sigma')
plt.ylabel('Count')
plt.title('Log-Normal Sigma Distribution');

In [ ]:
# Smoother visualization with KDE
import seaborn as sns

In [ ]:
# Kernel Density Estimate plot
sns.kdeplot(sig_samp, clip=(0, 10))
plt.xlabel('Sigma')
plt.ylabel('Density')
plt.title('Log-Normal Sigma Distribution (KDE)');

**Why log-normal?**

- Concentrates samples around medium noise levels (most informative for learning)
- Still covers both very low and very high noise levels
- Better than uniform because denoising difficulty varies non-linearly with sigma

### 🎮 Interactive: Watch a Training σ Being Born

The one-liner `(torch.randn([10000]) * 1.2 - 1.2).exp()` hides a three-stage pipeline. Chips **①–③** animate a single
draw riding through it — the dot travels the three panels while the matching code line lights up:

1. `z ~ N(0,1)`,  2. shift & scale into `log σ` (still a bell, recentered),  3. exponentiate — the symmetric bell in
log-space becomes the lopsided log-normal in σ-space, always positive.

Chip **④** builds the curriculum histogram draw by draw. Read it against the shaded bands: most mass lands in the teal
**sweet spot 0.1 < σ < 1** around σ_data, with principled tails into the trivial and hopeless zones. Drag the
**mean/std sliders** to see how the paper's (−1.2, 1.2) choice shapes where training effort goes.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_lognormal_sampler.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/karras_lognormal_sampler.html", height="950px")

---
## Noisify Function with Pre-conditioning

The noisify function now incorporates the Karras scalings and computes the appropriate training target.

In [ ]:
def noisify(x0):
    """
    Add noise with Karras pre-conditioning.
    
    Instead of training the network to predict noise (epsilon),
    we train it to predict a pre-conditioned target that accounts
    for the skip connection.
    
    Args:
        x0: Clean images
    
    Returns:
        ((scaled_noisy_input, sigma), target)
        - scaled_noisy_input: x_noisy * c_in (ready for network)
        - sigma: The noise level used
        - target: What the network should predict
    """
    device = x0.device
    
    # Sample sigma from log-normal distribution
    # Shape: (batch, 1, 1, 1) for broadcasting
    sig = (torch.randn([len(x0)]) * 1.2 - 1.2).exp().to(x0).reshape(-1, 1, 1, 1)
    # We are picking a sigma using log normal distribution
    
    # Generate noise
    noise = torch.randn_like(x0, device=device)
    
    # This is new now
    # Get pre-conditioning scalings
    c_skip, c_out, c_in = scalings(sig)
    
    # Create noisy input: x_noisy = x_clean + noise * sigma
    noised_input = x0 + noise * sig
    
    # Compute training target
    # The model output should satisfy: x_clean = c_skip * x_noisy + c_out * model_output
    # Solving for model_output: target = (x_clean - c_skip * x_noisy) / c_out
    target = (x0 - c_skip * noised_input) / c_out
    # Our target is not the original image, it is not the noise but it is
    # somewhere between the two 
    
    # Return scaled input (for network) and target
    return (noised_input * c_in, sig.squeeze()), target

**Key insight:**

The network doesn't predict noise directly. Instead, it predicts a **pre-conditioned target** that, when combined with the skip connection, gives the clean image:

```
x_clean = c_skip * x_noisy + c_out * F(c_in * x_noisy, sigma)
           ↑                  ↑       ↑
      skip connection    network output  scaled input
```

This formulation:
1. Naturally handles the skip connection
2. Keeps network inputs/outputs at unit variance
3. Makes the learning problem well-conditioned at all noise levels

### 🎮 Interactive: Step Through `noisify()` Line by Line

Everything so far — σ_data, the three dials, log-normal σ — meets inside this one function. The widget below runs the
exact same logic live on a toy image:

- **Click any stage chip ①–⑦** to jump straight there (its Python line lights up below), or press **Play** to watch the
  whole pipeline with the matching card glowing at each step.
- The first three panels share **one grayscale scale**, so the `+` and `=` between them are literally true on screen —
  at high σ the clean image visibly fades because the noise really does dwarf it.
- Drag the **σ override** (or hit **↺ new noise + σ** for a fresh log-normal draw) and watch the std read-outs: the raw
  noised image's std explodes with σ, but the **network input** and the **target** always land at std ≈ 1. That is
  pre-conditioning, verified before your eyes — the same check the notebook does numerically a few cells below.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_noisify_pipeline.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/karras_noisify_pipeline.html", height="1000px")

In [ ]:
def collate_ddpm(b):
    """Custom collate function."""
    return noisify(default_collate(b)[xl])


def dl_ddpm(ds):
    """Create DataLoader."""
    return DataLoader(ds, batch_size=bs, collate_fn=collate_ddpm, num_workers=8)

In [ ]:
# Create DataLoaders
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

In [ ]:
# Get a sample batch
dl = dls.train
(noised_input, sig), target = b = next(iter(dl))

In [ ]:
# Visualize noisy inputs (scaled by c_in)
show_images(noised_input[:25], imsize=1.5, titles=fc.map_ex(sig[:25], '{:.02f}'))

Notice the sigma values are now continuous (0.05, 0.23, 1.47, etc.) rather than integer timesteps.

In [ ]:
# Visualize targets (what the network should predict)
show_images(target[:25], imsize=1.5, titles=fc.map_ex(sig[:25], '{:.02f}'))

**Notice:**
- The targets look different from both the noisy images and the clean images
- This is the "residual" the network needs to learn
- The skip connection handles the easy part (passing through information)

In [ ]:
# Check that inputs and targets have reasonable statistics
# Both should have mean near 0 and std near 1
print(f"Noised input - mean: {noised_input.mean():.3f}, std: {noised_input.std():.3f}")
print(f"Target       - mean: {target.mean():.3f}, std: {target.std():.3f}")

Both inputs and targets have std close to 1 - this is the pre-conditioning working!

## Jeremy's Note:

In sampling, we should jump by big steps early on and small steps later on and make sure that the fine details are just so.

---
## Training

In [ ]:
class UNet(UNet2DModel):
    """UNet wrapper that unpacks inputs and extracts .sample from output."""
    def forward(self, x):
        return super().forward(*x).sample

In [ ]:
def init_ddpm(model):
    """Initialize UNet for stable training."""
    for o in model.down_blocks:
        for p in o.resnets:
            p.conv2.weight.data.zero_()
            for p in fc.L(o.downsamplers):
                init.orthogonal_(p.conv.weight)

    for o in model.up_blocks:
        for p in o.resnets:
            p.conv2.weight.data.zero_()

    model.conv_out.weight.data.zero_()

In [ ]:
# Training setup
lr = 1e-2
epochs = 25
opt_func = partial(optim.Adam, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)

cbs = [
    DeviceCB(),
    MixedPrecision(),
    ProgressCB(plot=True),
    MetricsCB(),
    BatchSchedCB(sched)
]

# Create model
model = UNet(
    in_channels=1,
    out_channels=1,
    block_out_channels=(32, 64, 128, 256),
    norm_num_groups=8
)
init_ddpm(model)

# Create Learner
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [ ]:
# Train
learn.fit(epochs)

In [ ]:
# Save/load model
# torch.save(learn.model, 'models/fashion_karras.pkl')
# model = learn.model = torch.load('models/fashion_karras.pkl', weights_only=False).cuda()

---
## Denoising with Pre-conditioning

To get the clean image from the network output, we apply the inverse of our pre-conditioning.

In [ ]:
def denoise(target, noised_input):
    """
    Convert network output back to clean image estimate.
    
    The model predicts: target = (x_clean - c_skip * x_noisy) / c_out
    So: x_clean = target * c_out + c_skip * x_noisy
    
    Note: noised_input here should be the UNSCALED noisy image
    (i.e., x_noisy, not x_noisy * c_in)
    """
    return target * c_out + noised_input * c_skip

### 🎮 Interactive 3D: Ride the EDM Wrapper

The `denoise()` inversion you just read — and the sampling-time wrapper coming later — are the same machine:
`denoised = c_skip·x + c_out·model(c_in·x, σ)`. Here it is as a **physical contraption you can orbit** (drag to rotate,
scroll to zoom):

- The orange tensor takes the low road through the **×c_in gate → network → ×c_out gate**, while a teal copy rides the
  **skip highway** over the top — the highway's *thickness is literally c_skip*.
- Drag **σ** and watch the machine re-rig itself: gate rings resize, the highway fattens or starves. Chip **⑤** sweeps σ
  end-to-end — at σ→0 the network only whispers corrections past a fat highway; at σ→80 the highway is a thread and the
  network does everything.
- **▶ Play trip** sends the tensor through once, with its label morphing (`x → c_in·x → F → c_out·F → denoised`) and the
  matching code line lit at each leg. Click any part — boxes, gates, the highway, the σ dial — for its live value.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_edm_blend_3d.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# NOTE: needs internet access (loads three.js from cdnjs).
# ============================================================================
show_viz("interactive_viz/karras_edm_blend_3d.html", height="1000px")

In [ ]:
# Test denoising on the batch
with torch.no_grad():
    # Reshape sigma for broadcasting
    sigr = sig.cuda().reshape(-1, 1, 1, 1)
    
    # Get scalings for these sigmas
    c_skip, c_out, c_in = scalings(sigr)
    
    # Get network prediction
    targ_pred = learn.model((noised_input.cuda(), sig.cuda()))
    
    # Denoise: note we divide by c_in to get unscaled noisy input
    x0_pred = denoise(targ_pred, noised_input.cuda() / c_in)

In [ ]:
# Show noisy inputs
show_images(noised_input[:25], imsize=1.5, titles=fc.map_ex(sig[:25], '{:.02f}'))

In [ ]:
# Show denoised predictions
show_images(x0_pred[:25].clamp(-1, 1), imsize=1.5, titles=fc.map_ex(sig[:25], '{:.02f}'))

The model can denoise images at various noise levels!

In [ ]:
# Show ground truth denoised (using actual targets)
show_images(denoise(target.cuda(), noised_input.cuda() / c_in)[:25], 
            imsize=1.5, titles=fc.map_ex(sig[:25], '{:.02f}'))

---
## Testing at High Noise

Let's test the model starting from pure noise (sigma = 80).

In [ ]:
# Very high noise level
sig_r = tensor(80.).cuda().reshape(-1, 1, 1, 1)

# Get scalings
c_skip, c_out, c_in = scalings(sig_r)

# Generate pure noise scaled by sigma
x_r = torch.randn(32, 1, 32, 32).to(model.device) * sig_r

# Predict and denoise
with torch.no_grad():
    targ_pred = learn.model((x_r * c_in, sig_r.squeeze()))
    x0_pred = denoise(targ_pred, x_r)

# Display
show_images(x0_pred[:25], imsize=1.5)

With one step from pure noise, the outputs are blurry but show the general structure the model learned.

In [ ]:
# Check statistics
print(f"Max: {x0_pred.max():.3f}, Min: {x0_pred.min():.3f}")
print(f"Mean: {x0_pred.mean():.3f}, Std: {x0_pred.std():.3f}")

---
## Sampling

Now let's implement proper multi-step sampling with various algorithms.

In [ ]:
from miniai.fid import ImageEval

In [ ]:
# Set up FID evaluation
cmodel = torch.load('models/data_aug.pkl', weights_only=False)
# This model trained on data has pixels between -1 and 1
del(cmodel[8])
del(cmodel[7])

bs = 2048
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=fc.defaults.cpus)

dt = dls.train
xb, yb = next(iter(dt))

ie = ImageEval(cmodel, dls, cbs=[DeviceCB()])
# But the dataloader we are passing has pixels between -0.5 and 0.5

In [ ]:
# Sample sizes
# sz = (512, 1, 32, 32)   # For quick testing
sz = (2048, 1, 32, 32)  # For FID evaluation

---
### Karras Sigma Schedule

The EDM paper proposes an optimal schedule of sigma values for sampling.

In [ ]:
def sigmas_karras(n, sigma_min=0.01, sigma_max=80., rho=7., device='cpu'):
    """
    Karras sigma schedule for sampling.
    
    This creates a sequence of sigma values that are spaced
    optimally according to the EDM paper. The spacing is
    denser at lower sigma (where details matter more).
    
    Args:
        n: Number of steps
        sigma_min: Minimum sigma (near-clean)
        sigma_max: Maximum sigma (near-noise)
        rho: Controls the spacing (7 is recommended)
        device: Device to place the tensor
    
    Returns:
        Tensor of sigma values from sigma_max to 0
    """
    # Linear ramp from 0 to 1
    ramp = torch.linspace(0, 1, n)
    
    # Inverse-rho transform of min and max
    min_inv_rho = sigma_min ** (1/rho)
    max_inv_rho = sigma_max ** (1/rho)
    
    # Interpolate in inverse-rho space, then transform back
    sigmas = (max_inv_rho + ramp * (min_inv_rho - max_inv_rho)) ** rho
    
    # Append 0 at the end (clean image)
    return torch.cat([sigmas, tensor([0.])]).to(device)

**Understanding the Karras schedule:**

```
With rho=7:
- Sigma values are NOT linearly spaced
- More steps at LOW sigma (where fine details matter)
- Fewer steps at HIGH sigma (where we're just removing bulk noise)

Example with 10 steps:
80 → 30 → 12 → 5 → 2.5 → 1.2 → 0.6 → 0.3 → 0.1 → 0.01 → 0
└──────────────────┘  └───────────────────────────────────┘
      Fewer steps              More steps (denser)
```

In [ ]:
# Visualize the schedule
sk = sigmas_karras(100)
plt.plot(sk)
plt.xlabel('Step')
plt.ylabel('Sigma')
plt.title('Karras Sigma Schedule');

The curve drops quickly at first (high sigma → medium sigma), then more gradually (medium → low sigma).

### 🎮 Interactive: Where Should the Sampling Steps Go?

The static plot above shows *one* schedule. The explorer below lets you ask the real question: **how does ρ trade
big early strides against tiny final steps?**

- Drag **ρ** from 1 (recovers plain linear spacing — compare with the grey dashed curve) up to 15 and watch the dots on
  the log-σ ruler migrate toward σ_min, the fine-detail zone.
- **Click any dot** to read its exact σ and how much noise (Δσ) that single step removes — with the defaults, the first
  steps do most of the heavy lifting while dozens of tiny final steps polish detail (the Δσ bar chart makes this vivid).
- The **①–④ chips** trace how the four lines of `sigmas_karras` build the curve — the orange dashed ghost shows the
  intermediate quantity at each stage: even spacing in σ^(1/ρ)-space is what becomes detail-friendly spacing in σ-space.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_schedule_explorer.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/karras_schedule_explorer.html", height="1000px")

---
### Denoise Function for Sampling

In [ ]:
def denoise(model, x, sig):
    """
    Denoise x at noise level sig using the pre-conditioned model.
    
    Args:
        model: The trained network
        x: Noisy image
        sig: Current noise level (sigma)
    
    Returns:
        Denoised image estimate
    """
    # Get scalings for this sigma
    c_skip, c_out, c_in = scalings(sig)
    
    # Apply pre-conditioning: D(x) = c_skip * x + c_out * F(c_in * x)
    return model((x * c_in, sig)) * c_out + x * c_skip

### 🎮 Interactive: The Arrow Field Every Sampler Follows

Before meeting any sampler, meet the thing they all consume: at every point, `d = (x − denoised) / σ` defines a pull.
The data here is a 2-D toy (three teal clusters standing in for image-space) so the **entire field is visible at once**:

- **Click anywhere** to dissect the arrow at that point: your position x, the model's clean-estimate D(x,σ) (orange ✕ —
  computed exactly for the toy, playing the trained network's role), and the resulting move direction, with live numbers.
- **Drag σ** and watch the field morph between the two regimes the chips replay: at high σ, one blurry magnet (every
  arrow points at the data's center of mass — why the first giant stride is safe); at low σ, three sharp magnets with
  real decision boundaries (why late steps must be tiny).
- Chip **④** releases 60 walkers down a 48-rung Karras ladder — *following the field is sampling*, and their two-phase
  motion (consensus, then commitment) is exactly what the next visualizations show as trajectories.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_ode_flow_field.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/karras_ode_flow_field.html", height="1050px")

---
### Euler Sampler (Basic)

The simplest ODE solver for diffusion sampling.

In [ ]:
@torch.no_grad()
def sample_euler(x, sigs, i, model):
    """
    One step of Euler sampling.
    
    Euler method: x_{i+1} = x_i + f(x_i) * dt
    
    For diffusion: the "velocity" is (x - denoised) / sigma
    
    Args:
        x: Current noisy image
        sigs: Array of sigma values
        i: Current step index
        model: Denoising model
    
    Returns:
        Updated image at next sigma level
    """
    sig, sig2 = sigs[i], sigs[i+1]  # Current and next sigma
    
    # Get denoised estimate
    denoised = denoise(model, x, sig)
    
    # Euler step: move along the direction from x to denoised
    # The step size is proportional to (sig2 - sig)
    return x + (x - denoised) / sig * (sig2 - sig)

**Euler method intuition:**

```
Current state: x at noise level sigma
Target: denoised (our best guess of the clean image)
Direction: (x - denoised) / sigma  (normalized difference)
Step size: sigma_next - sigma  (change in noise level)

x_next = x + direction * step_size
```

### 🎮 Interactive 3D: Down the Noise Mountain

Euler's update, staged as a physical descent you can orbit (drag to rotate, scroll to zoom). **Height = log σ; the floor
is data space.** Rings are the rungs of `sigmas_karras` — watch their spacing collapse toward the floor.

- Chip **③ Anatomy of one stride** is the key one: freeze on the hero ball and step ⏭ rung by rung — the model plants a
  purple **flag on the floor** (its clean-image guess D(x,σ)), the arrow shows the stride, and the three highlighted
  code lines match the three sub-moves. Early flags point at the blur *between* clusters; late flags commit.
- Chip **④** plays the full 26-particle descent: a plunge through the wide upper rings while everyone drifts centerward,
  then the fan-out into three streams. Chip **⑤** runs the hero on **Heun** against a red plain-Euler ghost — the gap
  between them *is* the discretization error Heun buys back with its second model call.
- Click any **ring** for that rung's numbers, click any **ball** to adopt a new hero, use the **rung pills** to jump,
  and re-space the ladder live with the **rungs** slider.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_sampling_3d.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# NOTE: needs internet access (loads three.js from cdnjs).
# ============================================================================
show_viz("interactive_viz/karras_sampling_3d.html", height="1100px")

---
### Euler Ancestral Sampler (Stochastic)

Adds noise at each step for more diversity.

In [ ]:
def get_ancestral_step(sigma_from, sigma_to, eta=1.):
    """
    Compute sigma_down and sigma_up for ancestral sampling.
    
    The idea: instead of going directly from sigma_from to sigma_to,
    we go to a lower sigma (sigma_down) then add noise (sigma_up)
    to reach sigma_to.
    
    Args:
        sigma_from: Starting sigma
        sigma_to: Target sigma
        eta: Controls how much noise to add (0=deterministic, 1=full)
    
    Returns:
        (sigma_down, sigma_up)
    """
    if not eta:
        return sigma_to, 0.
    
    var_to, var_from = sigma_to**2, sigma_from**2
    
    # Amount of noise to add
    sigma_up = min(sigma_to, eta * (var_to * (var_from - var_to) / var_from)**0.5)
    
    # Sigma to denoise to before adding noise
    sigma_down = (var_to - sigma_up**2)**0.5
    
    return sigma_down, sigma_up

In [ ]:
@torch.no_grad()
def sample_euler_ancestral(x, sigs, i, model, eta=1.):
    """
    One step of Euler Ancestral sampling.
    
    Like Euler, but adds noise after each step.
    This produces more diverse outputs.
    """
    sig, sig2 = sigs[i], sigs[i+1]
    
    # Get denoised estimate
    denoised = denoise(model, x, sig)
    
    # Compute ancestral step parameters
    sigma_down, sigma_up = get_ancestral_step(sig, sig2, eta=eta)
    
    # Euler step to sigma_down
    x = x + (x - denoised) / sig * (sigma_down - sig)
    
    # Add noise
    return x + torch.randn_like(x) * sigma_up

> **🎮 Try it now:** in the Sampler Playground (after the comparison table below), select **Euler Ancestral** and
> watch the **orange flashes** — that's the `torch.randn_like(x) * sig_up` line firing, and the code panel highlights the
> split-the-rung and re-noise lines as it happens. Drag **η** to 0 mid-run and the sampler degenerates into plain Euler
> (σ_up → 0, no noise re-added); at η = 1 the trails turn jittery and endpoints spread out — the diversity ancestral
> sampling buys, paid for in reproducibility.

---
### Heun Sampler (2nd Order)

A more accurate ODE solver that evaluates the model twice per step.

In [ ]:
@torch.no_grad()
def sample_heun(x, sigs, i, model, s_churn=0., s_tmin=0., s_tmax=float('inf'), s_noise=1.):
    """
    One step of Heun's method (improved Euler).
    
    Heun's method:
    1. Make a preliminary Euler step
    2. Evaluate the direction at the new point
    3. Average the two directions
    4. Take the final step using the averaged direction
    
    This is a 2nd-order method (more accurate than Euler).
    
    Args:
        x: Current image
        sigs: Sigma schedule
        i: Current step index
        model: Denoising model
        s_churn: Amount of noise to add (for diversity)
        s_tmin, s_tmax: Sigma range for churning
        s_noise: Noise multiplier
    """
    sig, sig2 = sigs[i], sigs[i+1]
    n = len(sigs)
    
    # Optionally add noise ("churn") for diversity
    gamma = min(s_churn / (n-1), 2**0.5 - 1) if s_tmin <= sig <= s_tmax else 0.
    eps = torch.randn_like(x) * s_noise
    sigma_hat = sig * (gamma + 1)
    if gamma > 0:
        x = x + eps * (sigma_hat**2 - sig**2)**0.5
    
    # First evaluation: get direction at current point
    denoised = denoise(model, x, sig)
    d = (x - denoised) / sig  # Direction
    dt = sig2 - sigma_hat     # Step size
    
    # Preliminary step
    x_2 = x + d * dt
    
    # If we're at the last step, just return
    if sig2 == 0:
        return x_2
    
    # Second evaluation: get direction at new point
    denoised_2 = denoise(model, x_2, sig2)
    d_2 = (x_2 - denoised_2) / sig2
    
    # Average the directions
    d_prime = (d + d_2) / 2
    
    # Final step with averaged direction
    return x + d_prime * dt

> **🎮 Try it now:** switch the Playground (below) to **Heun** and step one rung at a time: you'll see the
> **red ghosts** appear at the provisional Euler landing while the code panel sits on the *second* `denoise` call —
> then the corrected jump replaces them. That second call is the 2× cost printed in the code comment, which is why the
> honest comparison in the FID table is Heun-n against Euler-2n.

---
### Main Sampling Function

In [ ]:
def sample(sampler, model, steps=100, sigma_max=80., **kwargs):
    """
    Generate samples using the specified sampler.
    
    Args:
        sampler: Sampling function (sample_euler, sample_heun, etc.)
        model: Trained model
        steps: Number of sampling steps
        sigma_max: Starting noise level
        **kwargs: Additional arguments for the sampler
    
    Returns:
        List of intermediate predictions
    """
    preds = []
    
    # Start from pure noise scaled by sigma_max
    x = torch.randn(sz).to(model.device) * sigma_max
    
    # Get sigma schedule
    sigs = sigmas_karras(steps, device=model.device, sigma_max=sigma_max)
    
    # Iterate through steps
    for i in progress_bar(range(len(sigs) - 1)):
        x = sampler(x, sigs, i, model, **kwargs)
        preds.append(x)
    
    return preds

---
## Comparing Samplers

In [ ]:
# Euler sampler with 100 steps
preds = sample(sample_euler, model, steps=100)

# Get final samples
s = preds[-1]
print(f"Min: {s.min():.3f}, Max: {s.max():.3f}")

In [ ]:
# Display samples
show_images(s[:25].clamp(-1, 1), imsize=1.5)

In [ ]:
# Evaluate FID
print(f"Euler 100 steps - FID: {ie.fid(s):.2f}, KID: {ie.kid(s):.4f}")

### Sampler Comparison Results

Different samplers have different trade-offs:

| Sampler | Steps | FID | Notes |
|---------|-------|-----|-------|
| Euler | 100 | ~5.2-5.4 | Fast, simple |
| Euler Ancestral | 100 (eta=0.5) | ~5.5 | More diversity |
| Heun | 50 | ~6.2 | 2nd order, fewer steps |
| Heun | 20 | ~5.6 | Very fast |
| Heun + churn | 20 | ~5.3 | Fast + diverse |
| LMS | 20 | ~5.1 | Multi-step method |
| Real images | - | ~2.6 | Reference |

### 🎮 Interactive: The Sampler Playground — Four Ways Down the Same Mountain

Now that the table has named the trade-offs, watch them happen. The teal blobs are a 3-cluster toy dataset whose ideal
denoiser D(x, σ) has a closed form, so this is genuine diffusion sampling — same denoiser, same Karras ladder, only the
stepping policy changes:

- Press **▶ Play** (default speed is deliberately slow — raise the slider any time), or use **⏵ Substep** to advance
  exactly one code line per click, fully self-paced. Either way the executing line lights up on the right
  while the **purple bar under the canvas narrates it in plain words** and the canvas acts it out (bold purple arrows =
  the model's clean guesses, dashed arrows = the planned hop, **red dots** = Heun's provisional trial landings, **orange
  rings** = Ancestral's fresh-noise injections). The ⭐ hero particle gets its own labels; the key under the canvas
  decodes every mark.
- **Click anywhere on the σ progress bar** to jump the whole population to that stage of the descent, and click the
  canvas to drop extra particles.
- Switch chips as you read the cells above: **Ancestral** with η dragged between 0 (collapses to Euler) and 1; **Heun**
  with 4–6 steps to see trial-then-correct beat straight hops, plus the churn knob; **LMS** order 1 (= Euler) vs 4.
- When a run finishes, the stat line reports **where the samples landed vs the true 40/35/25% cluster weights** — a
  tiny visual FID.

In [ ]:
# ============================================================================
# INTERACTIVE -- run this cell. It loads ./interactive_viz/karras_sampler_playground.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Keep the 'interactive_viz' folder in the SAME directory as this notebook.
# ============================================================================
show_viz("interactive_viz/karras_sampler_playground.html", height="1100px")

---
## LMS Sampler (Linear Multi-Step)

A more sophisticated sampler that uses information from multiple previous steps.

In [ ]:
from scipy import integrate

def linear_multistep_coeff(order, t, i, j):
    """
    Compute coefficients for linear multi-step method.
    
    These coefficients determine how to combine information from
    the current step and previous steps.
    
    Args:
        order: Order of the method (how many previous steps to use)
        t: Array of sigma values
        i: Current step
        j: Index for the coefficient
    
    Returns:
        Coefficient value
    """
    if order - 1 > i:
        raise ValueError(f'Order {order} too high for step {i}')
    
    def fn(tau):
        prod = 1.
        for k in range(order):
            if j == k:
                continue
            prod *= (tau - t[i-k]) / (t[i-j] - t[i-k])
        return prod
    
    # Integrate to get coefficient
    return integrate.quad(fn, t[i], t[i+1], epsrel=1e-4)[0]

In [ ]:
@torch.no_grad()
def sample_lms(model, steps=100, order=4, sigma_max=80.):
    """
    Linear Multi-Step (LMS) sampler.
    
    Uses information from multiple previous steps to make
    more accurate predictions. Higher order = more accurate
    but requires more history.
    
    Args:
        model: Trained model
        steps: Number of steps
        order: Order of the method (typically 3-4)
        sigma_max: Starting noise level
    
    Returns:
        List of predictions
    """
    preds = []
    x = torch.randn(sz).to(model.device) * sigma_max
    sigs = sigmas_karras(steps, device=model.device, sigma_max=sigma_max)
    
    # Store derivative history
    ds = []
    
    for i in progress_bar(range(len(sigs) - 1)):
        sig = sigs[i]
        
        # Get denoised estimate and compute derivative
        denoised = denoise(model, x, sig)
        d = (x - denoised) / sig
        ds.append(d)
        
        # Keep only the last 'order' derivatives
        if len(ds) > order:
            ds.pop(0)
        
        # Compute coefficients
        cur_order = min(i + 1, order)
        coeffs = [linear_multistep_coeff(cur_order, sigs, i, j) for j in range(cur_order)]
        
        # Update x using weighted combination of derivatives
        x = x + sum(coeff * d for coeff, d in zip(coeffs, reversed(ds)))
        preds.append(x)
    
    return preds

---
## Summary

### Karras Pre-conditioning

1. **sigma_data**: Characteristic scale of your data (~0.5-1.0 for normalized images)

2. **Scalings**: c_skip, c_out, c_in normalize network inputs/outputs
   ```
   D(x, sigma) = c_skip * x + c_out * F(c_in * x, sigma)
   ```

3. **Log-normal sigma sampling**: Better than uniform for training

4. **Pre-conditioned target**: Network predicts residual, not noise directly

### Karras Sigma Schedule

- Non-uniform spacing: denser at low sigma
- Controlled by rho parameter (default: 7)
- Better than linear or cosine for many cases

### Samplers

| Sampler | Order | Evals/Step | Best For |
|---------|-------|------------|----------|
| Euler | 1 | 1 | Fast, simple |
| Euler Ancestral | 1 | 1 | Diversity |
| Heun | 2 | 2 | Accuracy |
| LMS | 3-4 | 1 | High accuracy |

---
## Next Steps

- **Classifier-free guidance**: Condition on class labels
- **DPM-Solver**: Even faster high-order samplers
- **Latent diffusion**: Work in compressed space
- **Higher resolution**: Apply to larger images